# Vision Metrics

Evaluate a Vision Language Model (VLM) using two metrics based on semantic similarity between the model's description and the human ground truth.

| Metric | What it tells you |
|--------|-------------------|
| **VisionSimilarity** | How semantically close the VLM descriptions are to the ground truth on average. A score of 1.0 means identical meaning; 0.0 means completely unrelated. |
| **VisionHallucination** | How often the VLM describes something significantly different from what actually happened. A frame is a hallucination when similarity falls below `threshold`. |

In [ ]:
!pip install alquimia-fair-forge

In [1]:
import logging
from fair_forge.metrics.vision import VisionSimilarity, VisionHallucination

logging.getLogger('httpx').disabled = True

c:\Users\ericf\Desktop\Alquimia\fair-forge\.venv\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


## Setup

Implementá un `Retriever` que cargue tu dataset de evaluación.

Cada `Batch` necesita solo dos campos de texto:
- `assistant` — descripción libre del VLM sobre el frame
- `ground_truth_assistant` — descripción humana de lo que realmente ocurrió

El `threshold` (por defecto `0.75`) define a partir de qué similitud una descripción se considera hallucination. Ajustalo según tu dominio.

In [2]:
import json
from pathlib import Path
from fair_forge import Retriever
from fair_forge.schemas import Dataset


class VLMRetriever(Retriever):
    def load_dataset(self) -> list[Dataset]:
        with open(Path('dataset.json'), encoding='utf-8') as f:
            return [Dataset.model_validate(item) for item in json.load(f)]


# dataset.json — estructura de cada evento:
#
# {
#   "session_id": "cam-lobby-01",
#   "assistant_id": "argos-gpt4o",
#   "language": "english",
#   "context": "Lobby entrance camera",
#   "conversation": [
#     {
#       "qa_id": "2026-03-13T14:00:00Z",
#       "query": "Describe what you observe in this frame.",
#       "assistant": "A person is lying on the floor near the entrance.",
#       "ground_truth_assistant": "A person fell near the entrance."
#     }
#   ]
# }

## Vision Similarity

How accurately does the VLM describe scenes compared to the human ground truth?

In [3]:
similarity_results = VisionSimilarity.run(VLMRetriever, threshold=0.75)

for r in similarity_results:
    print(r.summary)
    print()
    for i in r.interactions:
        print(f'  {i.qa_id}  similarity={i.similarity_score:.2f}')

2026-03-16 17:28:10,395 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: cpu
2026-03-16 17:28:10,395 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-mpnet-base-v2


cam-entrance-01:   0%|          | 0/5 [00:00<?, ?frame/s]

cam-parking-02:   0%|          | 0/4 [00:00<?, ?frame/s]

The model's descriptions have an average similarity of 59% with the ground truth (min: 28%, max: 89%) across 5 frames.

  frame_001  similarity=0.89
  frame_002  similarity=0.70
  frame_003  similarity=0.43
  frame_004  similarity=0.67
  frame_005  similarity=0.28
The model's descriptions have an average similarity of 74% with the ground truth (min: 33%, max: 90%) across 4 frames.

  frame_001  similarity=0.90
  frame_002  similarity=0.88
  frame_003  similarity=0.33
  frame_004  similarity=0.86


## Vision Hallucination

How often does the VLM describe something significantly different from what actually happened?

In [4]:
hallucination_results = VisionHallucination.run(VLMRetriever, threshold=0.75)

for r in hallucination_results:
    print(r.summary)
    print()
    for i in r.interactions:
        label = 'HALLUCINATION' if i.is_hallucination else 'ok'
        print(f'  {i.qa_id}  similarity={i.similarity_score:.2f}  {label}')

2026-03-16 17:28:17,017 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: cpu
2026-03-16 17:28:17,018 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-mpnet-base-v2


cam-entrance-01:   0%|          | 0/5 [00:00<?, ?frame/s]

cam-parking-02:   0%|          | 0/4 [00:00<?, ?frame/s]

The model hallucinated in 4 of 5 frames (80%). A frame is considered a hallucination when similarity with the ground truth falls below 0.75.

  frame_001  similarity=0.89  ok
  frame_002  similarity=0.70  HALLUCINATION
  frame_003  similarity=0.43  HALLUCINATION
  frame_004  similarity=0.67  HALLUCINATION
  frame_005  similarity=0.28  HALLUCINATION
The model hallucinated in 1 of 4 frames (25%). A frame is considered a hallucination when similarity with the ground truth falls below 0.75.

  frame_001  similarity=0.90  ok
  frame_002  similarity=0.88  ok
  frame_003  similarity=0.33  HALLUCINATION
  frame_004  similarity=0.86  ok
